# BERT-based Sarcasm Detection

Этот блокнот основан на разработанной ранее базовой модели (TF-IDF + логистическая регрессия).

Цель:
- Уловить контекстное значение текста
- Улучшить эффективность обнаружения сарказма
- Сравнить результаты с базовой моделью

In [21]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

## Загрузка данных

Для ускорения экспериментов используется заранее подготовленный сэмпл датасета (~30k–100k строк).

Сэмпл был получен из исходного датасета Reddit Sarcasm.

In [3]:
df = pd.read_csv("./df_sample.csv")
df.head()

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment
0,1,"Good thing Soros is so rich, these guys must b...",zaures,politics,7,-1,-1,2016-11,2016-11-16 02:12:37,Anti-Trump protests not letting up for sixth s...,good thing soros is so rich these guys must be...
1,0,I'm also not being able to send NZB files from...,pienocake,usenet,1,1,0,2016-03,2016-03-17 03:23:56,Congratulations! SAB has always been a great p...,im also not being able to send nzb files from ...
2,1,Or Kanye West?,griffin852,Monstercat,2,2,0,2014-08,2014-08-13 00:10:39,Or Concept?,or kanye west
3,0,Fuck.,EL_SUPER_BEASTO,Flyers,3,3,0,2014-01,2014-01-15 23:55:51,Hartnell listed as day to day after blocking shot,fuck
4,0,"I think the suit does have some tech in it, si...",zoahporre,movies,2,2,0,2016-02,2016-02-11 18:44:38,"Ok, so either Batman's *normal* suit has some ...",i think the suit does have some tech in it sim...


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [5]:
tokens = tokenizer(df['comment'].iloc[0])
print(tokens)

{'input_ids': [101, 2204, 2518, 2061, 7352, 2003, 2061, 4138, 1010, 2122, 4364, 2442, 2022, 2437, 1037, 4288, 2006, 12253, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [6]:
tokenizer.convert_ids_to_tokens(tokens['input_ids'])

['[CLS]',
 'good',
 'thing',
 'so',
 '##ros',
 'is',
 'so',
 'rich',
 ',',
 'these',
 'guys',
 'must',
 'be',
 'making',
 'a',
 'killing',
 'on',
 'overtime',
 '.',
 '[SEP]']

In [7]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [8]:
df = df[df['clean_comment'].notna()]

In [9]:
print(type(df['clean_comment'].iloc[0]))
print(df['clean_comment'].iloc[0])

<class 'str'>
good thing soros is so rich these guys must be making a killing on overtime


In [10]:
df_bert = df[['clean_comment', 'label']].copy()

df_bert = df_bert.rename(
    columns={'clean_comment': 'text',
             'label' : 'labels'}
    )

dataset = Dataset.from_pandas(df_bert)

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/99777 [00:00<?, ? examples/s]

In [11]:
dataset[0]

{'text': 'good thing soros is so rich these guys must be making a killing on overtime',
 'labels': 1,
 '__index_level_0__': 0,
 'input_ids': [101,
  2204,
  2518,
  2061,
  7352,
  2003,
  2061,
  4138,
  2122,
  4364,
  2442,
  2022,
  2437,
  1037,
  4288,
  2006,
  12253,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [12]:
train_df, test_df = train_test_split(
    df_bert,
    test_size=0.2,
    random_state=14
)

In [13]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

Map:   0%|          | 0/79821 [00:00<?, ? examples/s]

Map:   0%|          | 0/19956 [00:00<?, ? examples/s]

In [14]:
train_dataset.features

{'text': Value('string'),
 'labels': Value('int64'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

In [15]:
train_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

test_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=1,
    logging_dir="./logs",

    save_strategy="no"
    
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [22]:
trainer = Trainer(
    model=model,
    
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [23]:
trainer.train()

/Users/tryeno/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.558487,0.561287,0.716226,0.702652


TrainOutput(global_step=9978, training_loss=0.6023123766573572, metrics={'train_runtime': 6900.525, 'train_samples_per_second': 11.567, 'train_steps_per_second': 1.446, 'total_flos': 5250446887472640.0, 'train_loss': 0.6023123766573572, 'epoch': 1.0})

In [24]:
trainer.evaluate()

/Users/tryeno/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.558487,0.561287,1,0.716226,0.702652


{'eval_loss': 0.5612868070602417,
 'eval_accuracy': 0.7162256965323712,
 'eval_f1': 0.7026516145970071}

In [26]:
predictions = trainer.predict(test_dataset)

/Users/tryeno/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [27]:
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

In [28]:
results_df = pd.DataFrame({
    "text": test_df["text"].values,
    "true": y_true,
    "pred": y_pred
})

In [29]:
mistakes = results_df[results_df["true"] != results_df["pred"]]
mistakes.sample(10)

,text,true,pred
11355,because thats totally as probably as hillary c...,0,1
18042,jrmwise because hes a cringey fuck,1,0
16102,talking to therapist and thats why i can no lo...,0,1
5241,tim cahill tim cahill tim cahill tim cahill ti...,1,0
9557,it was,1,0
14570,maybe hell get some trump university vouchers,0,1
3588,cut the tip of a tube of caulking way back and...,1,0
11612,good to know that the democrats have nothing b...,0,1
14326,various bugs have been fixed stability improve...,1,0
5837,so accurate,0,1


### Наблюдения
* Модель, в отличии от вариации с TF-IDF, не клеймит большую часть коротких комментариев как автоматически саркастические
* Модель все еще испытывает трудности без контекста, ей явно не хватает упора на родительские комментарии
* Некоторые ошибки связаны с неоднозначьностью самих сообщений (например, модель не смогла вне контекста понять является фраза "so accurate" сарказмом или нет, пометив его как таковой по-умолчанию)


In [33]:
results_df['text'].iloc[14326]

'various bugs have been fixed stability improvements'

SyntaxError: invalid syntax (2986368851.py, line 1)